In [1]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "yellow-taxi-trips-2026"

In [2]:
# On charge l'extension BigQuery pour le notebook
%load_ext google.cloud.bigquery 

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\__init__.py:239: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


-- I/ Market Demand & Seasonality

-- Question 1: How does the demand for yellow taxis fluctuate over time (daily, weekly, monthly, and seasonally)?

In [ ]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.demand_over_time` AS
SELECT 
    DATE(tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM tpep_pickup_datetime) AS month,
    EXTRACT(WEEK FROM tpep_pickup_datetime) AS week,
    EXTRACT(DAYOFWEEK FROM tpep_pickup_datetime) AS weekday,
    COUNT(*) AS total_trips
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered`
GROUP BY trip_date, year, month, week, weekday
ORDER BY trip_date;

In [11]:
%%bigquery
SELECT * FROM `yellow-taxi-trips-2026.views_fordashboard.demand_over_time` LIMIT 10

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2714: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,trip_date,year,month,week,weekday,total_trips
0,2001-01-01,2001,1,0,2,11
1,2001-08-23,2001,8,33,5,1
2,2002-10-21,2002,10,42,2,36
3,2002-10-22,2002,10,42,3,53
4,2002-10-23,2002,10,42,4,53
5,2002-10-24,2002,10,42,5,51
6,2002-10-25,2002,10,42,6,53
7,2002-10-26,2002,10,42,7,36
8,2002-10-27,2002,10,43,1,46
9,2002-10-28,2002,10,43,2,25


-- Question 2: What are the peak hours for yellow taxi trips in different boroughs and zones? 

In [5]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.peak_hours_by_zone` AS
SELECT 
    EXTRACT(HOUR FROM t.tpep_pickup_datetime) AS pickup_hour,
    z.Borough,
    z.Zone,
    COUNT(*) AS total_trips
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` z
ON t.PULocationID = z.LocationID
GROUP BY pickup_hour, z.Borough, z.Zone
ORDER BY total_trips DESC;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [6]:
%%bigquery
SELECT * FROM `views_fordashboard.peak_hours_by_zone` LIMIT 100;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2714: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,pickup_hour,Borough,Zone,total_trips
0,18,Manhattan,Midtown Center,413561
1,17,Manhattan,Midtown Center,409262
2,14,Manhattan,Upper East Side South,389513
3,15,Manhattan,Upper East Side South,388813
4,16,Queens,JFK Airport,383311
...,...,...,...,...
95,10,Queens,LaGuardia Airport,210605
96,13,Manhattan,Midtown East,209844
97,18,Manhattan,East Chelsea,209536
98,20,Queens,LaGuardia Airport,209047


In [ ]:
%%bigquery
SELECT DISTINCT Zone FROM `views_fordashboard.peak_hours_by_zone`;

-- Question 3: How do weather conditions or major events (holidays, sports events) impact yellow taxi usage over time? (To be investigated later)


-- II/ Customer Behavior & Ride Characteristics. 

-- Question 4: What are the most popular pickup and drop-off locations, and how do they change over time?

In [7]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.popular_pickup_dropoff` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    EXTRACT(WEEK FROM t.tpep_pickup_datetime) AS week,
    EXTRACT(DAYOFWEEK FROM t.tpep_pickup_datetime) AS weekday,
    pz.Borough AS pickup_borough,
    pz.Zone AS pickup_zone,
    dz.Borough AS dropoff_borough,
    dz.Zone AS dropoff_zone,
    COUNT(*) AS total_trips
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` pz 
    ON t.PULocationID = pz.LocationID
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` dz 
    ON t.DOLocationID = dz.LocationID
GROUP BY trip_date, year, month, week, weekday, pickup_borough, pickup_zone, dropoff_borough, dropoff_zone;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [8]:
%%bigquery
SELECT * FROM `views_fordashboard.popular_pickup_dropoff` LIMIT 100;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2714: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,trip_date,year,month,week,weekday,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,total_trips
0,2024-04-30,2024,4,17,3,Queens,JFK Airport,Brooklyn,Downtown Brooklyn/MetroTech,13
1,2024-04-23,2024,4,16,3,Manhattan,Greenwich Village North,Manhattan,Hudson Sq,18
2,2024-04-12,2024,4,14,6,Manhattan,Upper East Side North,Manhattan,Lower East Side,15
3,2024-04-03,2024,4,13,4,Manhattan,Garment District,EWR,Newark Airport,7
4,2024-04-16,2024,4,15,3,Queens,JFK Airport,Bronx,Melrose South,5
...,...,...,...,...,...,...,...,...,...,...
95,2024-04-14,2024,4,15,1,Manhattan,Hamilton Heights,Manhattan,Times Sq/Theatre District,1
96,2024-04-03,2024,4,13,4,Manhattan,Hudson Sq,Manhattan,Upper East Side North,6
97,2024-04-01,2024,4,13,2,Queens,East Elmhurst,Brooklyn,Bay Ridge,3
98,2024-04-10,2024,4,14,4,Manhattan,Gramercy,Brooklyn,Crown Heights North,2


In [ ]:
%%bigquery
SELECT COUNT(*) FROM `views_fordashboard.popular_pickup_dropoff`;

-- Question 5: What is the average trip distance, and how does it vary by borough, time of day, and season?

In [9]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.avg_trip_distance_analysis` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    CASE 
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (12, 1, 2) THEN 'Winter'
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (3, 4, 5) THEN 'Spring'
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (6, 7, 8) THEN 'Summer'
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (9, 10, 11) THEN 'Fall'
    END AS season,
    EXTRACT(HOUR FROM t.tpep_pickup_datetime) AS pickup_hour,
    pz.Borough AS pickup_borough,
    dz.Borough AS dropoff_borough,
    AVG(t.trip_distance) AS avg_trip_distance
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` pz 
    ON t.PULocationID = pz.LocationID
JOIN `yellow-taxi-trips-2026.raw_yellowtrips.taxi_zone` dz 
    ON t.DOLocationID = dz.LocationID
GROUP BY trip_date, year, month, season, pickup_hour, pickup_borough, dropoff_borough
ORDER BY trip_date, pickup_hour;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [10]:
%%bigquery
SELECT * FROM `views_fordashboard.avg_trip_distance_analysis` LIMIT 100;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2714: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,trip_date,year,month,season,pickup_hour,pickup_borough,dropoff_borough,avg_trip_distance
0,2001-01-01,2001,1,Winter,0,Manhattan,Brooklyn,10.230000
1,2001-01-01,2001,1,Winter,0,Manhattan,Manhattan,3.323333
2,2001-01-01,2001,1,Winter,0,Queens,Unknown,18.630000
3,2001-01-01,2001,1,Winter,0,Queens,Manhattan,19.225000
4,2001-01-01,2001,1,Winter,1,Manhattan,Manhattan,3.890000
...,...,...,...,...,...,...,...,...
95,2002-10-24,2002,10,Fall,12,Manhattan,Manhattan,1.110000
96,2002-10-24,2002,10,Fall,12,Queens,Manhattan,9.450000
97,2002-10-24,2002,10,Fall,13,Queens,N/A,16.750000
98,2002-10-24,2002,10,Fall,13,Manhattan,Manhattan,1.240000


-- Question 6: How many trips have only one passenger versus multiple passengers, and does this change seasonally?

In [3]:
%%bigquery
CREATE OR REPLACE VIEW `views_fordashboard.passenger_trends_by_season` AS
SELECT 
    DATE(t.tpep_pickup_datetime) AS trip_date,
    EXTRACT(YEAR FROM t.tpep_pickup_datetime) AS year,
    EXTRACT(MONTH FROM t.tpep_pickup_datetime) AS month,
    CASE 
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (12, 1, 2) THEN 'Winter'
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (3, 4, 5) THEN 'Spring'
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (6, 7, 8) THEN 'Summer'
        WHEN EXTRACT(MONTH FROM t.tpep_pickup_datetime) IN (9, 10, 11) THEN 'Fall'
    END AS season,
    CASE 
        WHEN t.passenger_count = 1 THEN 'Single Passenger'
        ELSE 'Multiple Passengers'
    END AS passenger_category,
    COUNT(*) AS total_trips
FROM `yellow-taxi-trips-2026.transformed_data.cleaned_and_filtered` t
GROUP BY trip_date, year, month, season, passenger_category
ORDER BY trip_date;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)


""


In [4]:
%%bigquery
SELECT * FROM `views_fordashboard.passenger_trends_by_season` LIMIT 100;

c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\job\query.py:2170: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
c:\Users\EDITH ADJE\Documents\GCP Project\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2714: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,trip_date,year,month,season,passenger_category,total_trips
0,2001-01-01,2001,1,Winter,Single Passenger,9
1,2001-01-01,2001,1,Winter,Multiple Passengers,2
2,2001-08-23,2001,8,Summer,Single Passenger,1
3,2002-10-21,2002,10,Fall,Single Passenger,31
4,2002-10-21,2002,10,Fall,Multiple Passengers,5
...,...,...,...,...,...,...
95,2022-02-03,2022,2,Winter,Multiple Passengers,19266
96,2022-02-03,2022,2,Winter,Single Passenger,77413
97,2022-02-04,2022,2,Winter,Single Passenger,78822
98,2022-02-04,2022,2,Winter,Multiple Passengers,23000
